# Step 01_04_03 -- Minimal Cross-Dataset History View (aoe2companion)

**Phase:** 01 -- Data Exploration
**Pipeline Section:** 01_04 -- Data Cleaning
**Step:** 01_04_03
**Dataset:** aoe2companion (sibling of sc2egset PR #152 + aoestats)
**Predecessor:** 01_04_02 (Data Cleaning Execution -- complete)
**Step scope:** Create `matches_history_minimal` TABLE -- 8-column player-row-grain
projection of `matches_1v1_clean` (2 rows per 1v1 match, natively player-row-oriented).
Self-join pattern (sc2egset precedent) -- no UNION ALL needed (aoec is already 2-row
per match). Canonical TIMESTAMP temporal dtype (pass-through; matches_raw.started
is already TIMESTAMP). Per-dataset-polymorphic faction vocabulary (full civ names).
Cross-dataset-harmonized substrate for Phase 02+ rating-system backtesting.
Pure projection (I9).

**IMPLEMENTATION NOTE:** DuckDB 1.5.1 has a known internal bug where a self-join on
a VIEW that uses window functions with QUALIFY (such as matches_1v1_clean, which
contains row_number() + ANY() subquery) causes either an InternalException or
incorrect row counts. The workaround: (1) materialize the filtered base data into
a staging TABLE _mhm_base, (2) self-join _mhm_base to produce matches_history_minimal
as a persistent TABLE, (3) drop the staging table. The semantics are identical to
the plan's CREATE OR REPLACE VIEW intent -- same 8-column contract, same filtering.
The object_type is TABLE not VIEW; DESCRIBE produces identical schema.

**Invariants applied:**
  - I3 (TIMESTAMP pass-through; already TIMESTAMP in matches_raw.started)
  - I5-analog (player-row symmetry, NULL-safe assertion via IS DISTINCT FROM)
  - I6 (DDL + every assertion SQL stored verbatim in validation JSON artifact)
  - I7 (matchId INTEGER -> numeric-tail regex [0-9]+ with round-trip cast provenance)
  - I8 (8-column cross-dataset contract; per-dataset-polymorphic faction vocabulary)
  - I9 (pure non-destructive projection; no upstream modification)
**BLOCKER-1 (aoec):** matchId is INTEGER (variable decimal width). Fixed-length gate
inapplicable. Uses numeric-tail regex [0-9]+ + round-trip BIGINT cast assertion.
**BLOCKER-2 (aoec):** No slot column -- slot-bias gate is N/A (natively player-row).
I5-analog symmetry still enforced.
**Date:** 2026-04-18

## Cell 2 -- Imports

In [1]:
import json
from datetime import date
from pathlib import Path

import yaml

from rts_predict.common.notebook_utils import (
    get_notebook_db,
    get_reports_dir,
    setup_notebook_logging,
)

logger = setup_notebook_logging()
print("Imports complete.")

Imports complete.


## Cell 3 -- DuckDB Connection (writable -- creates TABLE)

This notebook creates one new TABLE: `matches_history_minimal`.
A writable connection is required.
WARNING: Close all read-only notebook connections to this DB before running.
Pre-execution constraint: no parallel CLI writes during T03.

In [2]:
db = get_notebook_db("aoe2", "aoe2companion", read_only=False)
con = db.con
print("DuckDB connection opened (read-write).")

DuckDB connection opened (read-write).


## Cell 4 -- Source-view sanity check

DESCRIBE matches_1v1_clean; assert 48 cols + presence of required columns.
Verifies the expected schema from matches_1v1_clean.yaml (01_04_02 artifact).
Required cols: matchId, started, profileId, civ, won (+ other 01_04_02 cols; 48 total).

In [3]:
describe_src = con.execute("DESCRIBE matches_1v1_clean").fetchall()
src_col_names = [row[0] for row in describe_src]
print(f"matches_1v1_clean column count: {len(src_col_names)}")
print(f"Columns: {src_col_names}")

# Assert 48 columns (per matches_1v1_clean.yaml step 01_04_02)
assert len(src_col_names) == 48, (
    f"Expected 48 columns in matches_1v1_clean, got {len(src_col_names)}"
)

# Assert required columns are present
required_cols = ["matchId", "started", "profileId", "civ", "won"]
for col in required_cols:
    assert col in src_col_names, (
        f"Required column '{col}' missing from matches_1v1_clean"
    )

print("Source-view sanity check PASSED: 48 cols + all required columns present.")

matches_1v1_clean column count: 48
Columns: ['matchId', 'started', 'leaderboard', 'name', 'internalLeaderboardId', 'privacy', 'map', 'difficulty', 'startingAge', 'fullTechTree', 'allowCheats', 'empireWarsMode', 'endingAge', 'gameMode', 'lockSpeed', 'lockTeams', 'mapSize', 'population', 'hideCivs', 'recordGame', 'regicideMode', 'gameVariant', 'resources', 'sharedExploration', 'speed', 'speedFactor', 'suddenDeathMode', 'civilizationSet', 'teamPositions', 'teamTogether', 'treatyLength', 'turboMode', 'victory', 'revealMap', 'profileId', 'rating', 'rating_was_null', 'color', 'colorHex', 'slot', 'team', 'won', 'country', 'shared', 'verified', 'civ', 'filename', 'is_null_cluster']
Source-view sanity check PASSED: 48 cols + all required columns present.


## Cell 5 -- Define DDL constants

Three-step DDL due to DuckDB 1.5.1 bug: self-join on matches_1v1_clean VIEW
(which contains row_number() window function + ANY() subquery) triggers
InternalException or returns wrong row counts. Additionally, a CTE self-join
(even with QUALIFY) inside a single CREATE TABLE AS statement also produces
wrong row counts (42,866 instead of 61,062,392). The only reliable approach
is three separate CREATE TABLE statements, each referencing the previous
persistent table (not a CTE):

Step 1: CREATE TABLE _good_match_ids -- good matchIds (complementary won pairs).
Step 2: CREATE TABLE _mhm_base -- base projection from matches_raw + _good_match_ids.
Step 3: CREATE TABLE matches_history_minimal -- self-join _mhm_base.
Step 4: DROP the two staging tables.

I7 cite: matchId INTEGER per data/db/schemas/raw/matches_raw.yaml.
I9: matches_raw is read-only; no modifications to upstream tables.

In [4]:
CREATE_GOOD_MATCH_IDS_SQL = """\
CREATE OR REPLACE TABLE _good_match_ids AS
-- Staging step 1/2: good match IDs (complementary won/lost pairs after dedup).
-- DuckDB 1.5.1 bug: QUALIFY + GROUP BY in a single CTE gives wrong row counts
-- when that CTE is used in a subsequent INNER JOIN. Materialized separately.
-- I9: matches_raw read-only; no upstream modification.
WITH deduped AS (
    SELECT matchId, profileId, won
    FROM matches_raw
    WHERE internalLeaderboardId IN (6, 18)
      AND profileId != -1
    QUALIFY
        row_number() OVER (PARTITION BY matchId, profileId ORDER BY started) = 1
)
SELECT matchId
FROM deduped
GROUP BY matchId
HAVING COUNT(*) = 2
   AND SUM(CASE WHEN won THEN 1 ELSE 0 END) = 1
   AND SUM(CASE WHEN NOT won THEN 1 ELSE 0 END) = 1\
"""

CREATE_STAGING_TABLE_SQL = """\
CREATE OR REPLACE TABLE _mhm_base AS
-- Staging step 2/2: base projection from matches_raw + _good_match_ids.
-- I7: matchId INTEGER per data/db/schemas/raw/matches_raw.yaml line `matchId: INTEGER`.
-- I9: matches_raw read-only; no upstream modification.
SELECT
    'aoe2companion::' || CAST(r.matchId AS VARCHAR) AS match_id,
    r.started                                       AS started_at,
    CAST(r.profileId AS VARCHAR)                    AS player_id,
    r.civ                                           AS faction,
    r.won                                           AS won
FROM matches_raw r
INNER JOIN _good_match_ids g ON r.matchId = g.matchId
WHERE r.internalLeaderboardId IN (6, 18)
  AND r.profileId != -1
QUALIFY
    row_number() OVER (PARTITION BY r.matchId, r.profileId ORDER BY r.started) = 1\
"""

CREATE_MATCHES_HISTORY_MINIMAL_SQL = """\
CREATE OR REPLACE TABLE matches_history_minimal AS
-- aoe2companion sibling of sc2egset.matches_history_minimal (PR #152 pattern).
-- Input: _mhm_base (materialized staging from matches_raw with matches_1v1_clean
--   filter logic). Strategy: self-join on match_id with unequal player_id
--   (sc2egset pattern). No UNION ALL pivot needed -- aoec already 2 rows/match.
-- Grain: 2 rows per 1v1 match (player row + opponent row, symmetric swap).
-- Cross-dataset contract: 8 columns, identical dtypes across sibling objects.
--   Canonical temporal dtype = TIMESTAMP (no TZ). Faction vocabulary is
--   per-dataset-polymorphic (SC2 race stems vs AoE2 full civ names).
-- Invariants: I3 (TIMESTAMP pass-through; started already TIMESTAMP per
--   data/db/schemas/raw/matches_raw.yaml), I5-analog (NULL-safe symmetry;
--   slot-bias gate N/A -- aoec natively player-row, no slot column),
--   I6 (DDL verbatim in JSON artifact), I7 (matchId INTEGER ->
--   numeric-tail regex [0-9]+; provenance: data/db/schemas/raw/matches_raw.yaml
--   line `matchId: INTEGER`), I8 (UNION-compatible with sibling datasets
--   via dataset_tag + prefixed match_id + canonical dtypes), I9 (pure
--   projection; no upstream modification).
SELECT
    p.match_id,
    p.started_at,
    p.player_id,
    o.player_id       AS opponent_id,
    p.faction,
    o.faction         AS opponent_faction,
    p.won,
    'aoe2companion'   AS dataset_tag
FROM _mhm_base p
JOIN _mhm_base o
  ON p.match_id = o.match_id
 AND p.player_id <> o.player_id
ORDER BY p.started_at, p.match_id, p.player_id\
"""

DROP_STAGING_TABLES_SQL = [
    "DROP TABLE IF EXISTS _mhm_base",
    "DROP TABLE IF EXISTS _good_match_ids",
]

print("DDL constants defined.")
print("\n=== STEP 1 DDL (good_match_ids staging) ===")
print(CREATE_GOOD_MATCH_IDS_SQL)
print("\n=== STEP 2 DDL (_mhm_base staging) ===")
print(CREATE_STAGING_TABLE_SQL)
print("\n=== STEP 3 DDL (matches_history_minimal) ===")
print(CREATE_MATCHES_HISTORY_MINIMAL_SQL)

DDL constants defined.

=== STEP 1 DDL (good_match_ids staging) ===
CREATE OR REPLACE TABLE _good_match_ids AS
-- Staging step 1/2: good match IDs (complementary won/lost pairs after dedup).
-- DuckDB 1.5.1 bug: QUALIFY + GROUP BY in a single CTE gives wrong row counts
-- when that CTE is used in a subsequent INNER JOIN. Materialized separately.
-- I9: matches_raw read-only; no upstream modification.
WITH deduped AS (
    SELECT matchId, profileId, won
    FROM matches_raw
    WHERE internalLeaderboardId IN (6, 18)
      AND profileId != -1
    QUALIFY
        row_number() OVER (PARTITION BY matchId, profileId ORDER BY started) = 1
)
SELECT matchId
FROM deduped
GROUP BY matchId
HAVING COUNT(*) = 2
   AND SUM(CASE WHEN won THEN 1 ELSE 0 END) = 1
   AND SUM(CASE WHEN NOT won THEN 1 ELSE 0 END) = 1

=== STEP 2 DDL (_mhm_base staging) ===
CREATE OR REPLACE TABLE _mhm_base AS
-- Staging step 2/2: base projection from matches_raw + _good_match_ids.
-- I7: matchId INTEGER per data/db/schemas/

## Cell 6 -- Execute DDL (create TABLE)

Four-step execution (DuckDB 1.5.1 workaround):
1. Create _good_match_ids TABLE (good matchIds: complementary won pairs).
2. Create _mhm_base TABLE (base projection: matches_raw INNER JOIN _good_match_ids).
3. Create matches_history_minimal TABLE (self-join _mhm_base).
4. Drop both staging tables (_mhm_base and _good_match_ids).

In [5]:
# Step 0: Drop any existing matches_history_minimal and staging tables
for obj in ["matches_history_minimal", "_mhm_base", "_good_match_ids"]:
    con.execute(f"DROP TABLE IF EXISTS {obj}")
    con.execute(f"DROP VIEW IF EXISTS {obj}")
print("Step 0: Cleaned up existing objects.")

# Step 1: good_match_ids staging table
con.execute(CREATE_GOOD_MATCH_IDS_SQL)
gm_count = con.execute("SELECT COUNT(*) FROM _good_match_ids").fetchone()[0]
print(f"Step 1: _good_match_ids created. Count: {gm_count}")
assert gm_count == 30_531_196, (
    f"good_match_ids count mismatch: expected 30_531_196, got {gm_count}"
)

# Step 2: _mhm_base staging table
con.execute(CREATE_STAGING_TABLE_SQL)
staging_count = con.execute("SELECT COUNT(*) FROM _mhm_base").fetchone()[0]
print(f"Step 2: _mhm_base created. Staging row count: {staging_count}")
assert staging_count == 61_062_392, (
    f"Staging count mismatch: expected 61_062_392, got {staging_count}"
)

# Step 3: matches_history_minimal
con.execute(CREATE_MATCHES_HISTORY_MINIMAL_SQL)
print("Step 3: TABLE matches_history_minimal created.")

# Step 4: Drop staging tables
for sql in DROP_STAGING_TABLES_SQL:
    con.execute(sql)
print("Step 4: Staging tables dropped.")

print("TABLE matches_history_minimal created successfully.")

Step 0: Cleaned up existing objects.


Step 1: _good_match_ids created. Count: 30531196


Step 2: _mhm_base created. Staging row count: 61062392


Step 3: TABLE matches_history_minimal created.
Step 4: Staging tables dropped.
TABLE matches_history_minimal created successfully.


## Cell 7 -- Schema shape validation

DESCRIBE matches_history_minimal; assert 8 columns + exact dtypes per spec.
Expected: [VARCHAR, TIMESTAMP, VARCHAR, VARCHAR, VARCHAR, VARCHAR, BOOLEAN, VARCHAR]
for columns [match_id, started_at, player_id, opponent_id, faction, opponent_faction, won, dataset_tag].
I3: started_at DTYPE must be TIMESTAMP (pass-through -- no cast failures expected).

In [6]:
describe_view = con.execute("DESCRIBE matches_history_minimal").fetchall()
view_col_names = [row[0] for row in describe_view]
view_col_types = [str(row[1]) for row in describe_view]

print(f"matches_history_minimal column count: {len(view_col_names)}")
for name, dtype in zip(view_col_names, view_col_types):
    print(f"  {name}: {dtype}")

# Assert 8 columns
assert len(view_col_names) == 8, (
    f"Expected 8 columns, got {len(view_col_names)}: {view_col_names}"
)

# Assert column names in order
expected_col_names = [
    "match_id", "started_at", "player_id", "opponent_id",
    "faction", "opponent_faction", "won", "dataset_tag",
]
assert view_col_names == expected_col_names, (
    f"Column name mismatch:\n  expected: {expected_col_names}\n  got:      {view_col_names}"
)

# Assert dtypes in order
expected_dtypes = [
    "VARCHAR", "TIMESTAMP", "VARCHAR", "VARCHAR",
    "VARCHAR", "VARCHAR", "BOOLEAN", "VARCHAR",
]
assert view_col_types == expected_dtypes, (
    f"Dtype mismatch:\n  expected: {expected_dtypes}\n  got:      {view_col_types}"
)

print("Schema shape validation PASSED: 8 cols + dtypes match spec.")

matches_history_minimal column count: 8
  match_id: VARCHAR
  started_at: TIMESTAMP
  player_id: VARCHAR
  opponent_id: VARCHAR
  faction: VARCHAR
  opponent_faction: VARCHAR
  won: BOOLEAN
  dataset_tag: VARCHAR
Schema shape validation PASSED: 8 cols + dtypes match spec.


## Cell 8 -- Row-count validation

Gate: total_rows=61_062_392; distinct_match_ids=30_531_196; matches_with_not_2_rows=0.
aoec matches_1v1_clean is natively 2-rows/match -> 30_531_196 matches x 2 = 61_062_392.

In [7]:
ROW_COUNT_CHECK_SQL = """\
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT match_id) AS distinct_match_ids
FROM matches_history_minimal\
"""

ROW_COUNT_SRC_SQL = """\
SELECT COUNT(*) AS src_rows FROM matches_1v1_clean\
"""

ROW_COUNT_NOT_2_SQL = """\
SELECT COUNT(*) AS matches_with_not_2_rows
FROM (
    SELECT match_id, COUNT(*) AS n
    FROM matches_history_minimal
    GROUP BY match_id
    HAVING n <> 2
)\
"""

rc = con.execute(ROW_COUNT_CHECK_SQL).fetchone()
total_rows, distinct_match_ids = rc

src_rows = con.execute(ROW_COUNT_SRC_SQL).fetchone()[0]
matches_with_not_2_rows = con.execute(ROW_COUNT_NOT_2_SQL).fetchone()[0]
matches_with_2_rows = distinct_match_ids  # since matches_with_not_2_rows = 0

print(f"total_rows:             {total_rows}")
print(f"distinct_match_ids:     {distinct_match_ids}")
print(f"src_rows:               {src_rows}")
print(f"matches_with_not_2_rows:{matches_with_not_2_rows}")

assert total_rows == 61_062_392, f"Expected total_rows=61_062_392, got {total_rows}"
assert distinct_match_ids == 30_531_196, f"Expected distinct_match_ids=30_531_196, got {distinct_match_ids}"
assert src_rows == 61_062_392, f"Expected src_rows=61_062_392, got {src_rows}"
assert matches_with_not_2_rows == 0, f"Expected matches_with_not_2_rows=0, got {matches_with_not_2_rows}"

print("Row-count validation PASSED.")

total_rows:             61062392
distinct_match_ids:     30531196
src_rows:               61062392
matches_with_not_2_rows:0
Row-count validation PASSED.


## Cell 9 -- Symmetry (I5-analog, NULL-safe)

Gate: symmetry_violations=0. Uses IS DISTINCT FROM for NULL-safe comparison.
Checks: (player_id, opponent_id) are swapped; won values are complementary;
faction and opponent_faction are mirrors.
NOTE: Slot-bias gate is N/A for aoec (natively player-row; no slot column).

In [8]:
SYMMETRY_I5_ANALOG_SQL = """\
WITH row_pairs AS (
    SELECT
        a.match_id,
        a.player_id         AS a_pid,
        a.opponent_id       AS a_oid,
        a.won               AS a_won,
        a.faction           AS a_fac,
        a.opponent_faction  AS a_ofac,
        b.player_id         AS b_pid,
        b.opponent_id       AS b_oid,
        b.won               AS b_won,
        b.faction           AS b_fac,
        b.opponent_faction  AS b_ofac
    FROM matches_history_minimal a
    JOIN matches_history_minimal b
      ON a.match_id = b.match_id
     AND a.player_id <> b.player_id
)
SELECT COUNT(*) AS symmetry_violations
FROM row_pairs
WHERE a_pid <> b_oid
   OR a_oid <> b_pid
   OR a_won = b_won
   OR a_fac IS DISTINCT FROM b_ofac
   OR a_ofac IS DISTINCT FROM b_fac\
"""

sym_row = con.execute(SYMMETRY_I5_ANALOG_SQL).fetchone()
symmetry_violations = sym_row[0]
print(f"symmetry_violations: {symmetry_violations}")

assert symmetry_violations == 0, (
    f"I5-analog NULL-safe symmetry violations: {symmetry_violations} (expected 0)"
)

print("Symmetry (I5-analog, NULL-safe) PASSED.")
print("NOTE: Slot-bias gate is N/A for aoec (natively player-row; no slot column).")

symmetry_violations: 0
Symmetry (I5-analog, NULL-safe) PASSED.
NOTE: Slot-bias gate is N/A for aoec (natively player-row; no slot column).


## Cell 10 -- Zero-NULL on non-nullable spec columns

Gate: null_match_id / null_player_id / null_opponent_id / null_won / null_dataset_tag all 0.
Gate: null_faction / null_opponent_faction all 0 (civ is zero-NULL upstream per
matches_1v1_clean.yaml notes -- stricter than sc2/aoestats).
started_at: also gate=0 (pass-through; no TRY_CAST failures since upstream started is TIMESTAMP).

In [9]:
ZERO_NULL_SQL = """\
SELECT
    COUNT(*) FILTER (WHERE match_id          IS NULL) AS null_match_id,
    COUNT(*) FILTER (WHERE started_at        IS NULL) AS null_started_at,
    COUNT(*) FILTER (WHERE player_id         IS NULL) AS null_player_id,
    COUNT(*) FILTER (WHERE opponent_id       IS NULL) AS null_opponent_id,
    COUNT(*) FILTER (WHERE won               IS NULL) AS null_won,
    COUNT(*) FILTER (WHERE dataset_tag       IS NULL) AS null_dataset_tag,
    COUNT(*) FILTER (WHERE faction           IS NULL) AS null_faction,
    COUNT(*) FILTER (WHERE opponent_faction  IS NULL) AS null_opponent_faction
FROM matches_history_minimal\
"""

null_row = con.execute(ZERO_NULL_SQL).fetchone()
(
    null_match_id, null_started_at, null_player_id, null_opponent_id,
    null_won, null_dataset_tag, null_faction, null_opponent_faction
) = null_row

print(f"null_match_id:          {null_match_id}")
print(f"null_started_at:        {null_started_at}  (gate=0; pass-through TIMESTAMP)")
print(f"null_player_id:         {null_player_id}")
print(f"null_opponent_id:       {null_opponent_id}")
print(f"null_won:               {null_won}")
print(f"null_dataset_tag:       {null_dataset_tag}")
print(f"null_faction:           {null_faction}  (gate=0; civ zero-NULL upstream)")
print(f"null_opponent_faction:  {null_opponent_faction}  (gate=0; civ zero-NULL upstream)")

assert null_match_id == 0, f"null_match_id={null_match_id} (expected 0)"
assert null_player_id == 0, f"null_player_id={null_player_id} (expected 0)"
assert null_opponent_id == 0, f"null_opponent_id={null_opponent_id} (expected 0)"
assert null_won == 0, f"null_won={null_won} (expected 0)"
assert null_dataset_tag == 0, f"null_dataset_tag={null_dataset_tag} (expected 0)"
assert null_faction == 0, f"null_faction={null_faction} (expected 0; civ zero-NULL upstream)"
assert null_opponent_faction == 0, f"null_opponent_faction={null_opponent_faction} (expected 0)"

print("Zero-NULL validation PASSED for all 7 non-nullable spec columns (incl. faction).")

null_match_id:          0
null_started_at:        0  (gate=0; pass-through TIMESTAMP)
null_player_id:         0
null_opponent_id:       0
null_won:               0
null_dataset_tag:       0
null_faction:           0  (gate=0; civ zero-NULL upstream)
null_opponent_faction:  0  (gate=0; civ zero-NULL upstream)
Zero-NULL validation PASSED for all 7 non-nullable spec columns (incl. faction).


## Cell 11 -- match_id prefix verification (aoec-specific: numeric-tail regex)

Gate: prefix_violations=0.
aoec-specific: matchId is INTEGER (variable decimal width) -- no fixed-length gate.
Checks: LIKE 'aoe2companion::%' AND numeric-tail regex [0-9]+ AND round-trip BIGINT cast.
I7 provenance: `matchId: INTEGER` in
src/rts_predict/games/aoe2/datasets/aoe2companion/data/db/schemas/raw/matches_raw.yaml.

In [10]:
PREFIX_CHECK_SQL = """\
-- Magic literal: 'aoe2companion::' = dataset prefix.
-- Numeric-tail regex [0-9]+ cites:
--   src/rts_predict/games/aoe2/datasets/aoe2companion/data/db/schemas/raw/matches_raw.yaml
--   line `matchId: INTEGER` (I7 provenance).
-- Round-trip cast: regexp_extract tail == CAST(CAST(tail AS BIGINT) AS VARCHAR)
--   verifies no leading zeros or non-numeric chars in match_id suffix.
-- Variable decimal width -- no fixed-length gate (unlike sc2egset's 42-char hex).
SELECT COUNT(*) AS prefix_violations
FROM matches_history_minimal m
WHERE m.match_id NOT LIKE 'aoe2companion::%'
   OR regexp_extract(m.match_id, '::([0-9]+)$', 1) = ''
   OR regexp_extract(m.match_id, '::([0-9]+)$', 1)
      <> CAST(CAST(split_part(m.match_id, '::', 2) AS BIGINT) AS VARCHAR)\
"""

prefix_row = con.execute(PREFIX_CHECK_SQL).fetchone()
prefix_violations = prefix_row[0]
print(f"prefix_violations: {prefix_violations}")

assert prefix_violations == 0, f"prefix_violations={prefix_violations} (expected 0)"

print("Prefix check (numeric-tail regex + round-trip cast) PASSED.")

prefix_violations: 0
Prefix check (numeric-tail regex + round-trip cast) PASSED.


## Cell 12 -- dataset_tag constant

Gate: n_distinct_tags=1, the_tag='aoe2companion'.

In [11]:
DATASET_TAG_CHECK_SQL = """\
SELECT
    COUNT(DISTINCT dataset_tag) AS n_distinct_tags,
    MAX(dataset_tag)            AS the_tag
FROM matches_history_minimal\
"""

tag_row = con.execute(DATASET_TAG_CHECK_SQL).fetchone()
n_distinct_tags, the_tag = tag_row
print(f"n_distinct_tags: {n_distinct_tags}")
print(f"the_tag:         {the_tag}")

assert n_distinct_tags == 1, f"n_distinct_tags={n_distinct_tags} (expected 1)"
assert the_tag == "aoe2companion", f"the_tag={the_tag!r} (expected 'aoe2companion')"

print("dataset_tag constant PASSED.")

n_distinct_tags: 1
the_tag:         aoe2companion
dataset_tag constant PASSED.


## Cell 13 -- Faction vocabulary (exploratory, no gate)

Documents per-dataset polymorphism (I8). aoe2companion ships full civ names
(e.g., Franks, Mongols, Britons, etc.) -- NOT 4-char stems like sc2egset.
No hard assertion -- exploratory per plan spec.

In [12]:
FACTION_VOCAB_SQL = """\
SELECT faction, COUNT(*) AS n
FROM matches_history_minimal
GROUP BY faction
ORDER BY n DESC\
"""

faction_rows = con.execute(FACTION_VOCAB_SQL).fetchall()
print("Faction vocabulary (per-dataset polymorphic, aoe2companion full civ names):")
for faction, n in faction_rows[:10]:
    print(f"  {faction!r}: {n}")
if len(faction_rows) > 10:
    print(f"  ... ({len(faction_rows) - 10} more)")

faction_vocab = {row[0]: row[1] for row in faction_rows}
print(f"\nTotal faction vocab size: {len(faction_vocab)} distinct values")
print("NOTE: aoe2companion faction vocabulary is full civilization names (Franks, Mongols, etc.).")
print("      Consumers MUST NOT treat faction as cross-dataset categorical without game-conditional encoding.")

Faction vocabulary (per-dataset polymorphic, aoe2companion full civ names):
  'franks': 3654980
  'mongols': 3637382
  'britons': 2307906
  'magyars': 2083367
  'spanish': 2049447
  'persians': 1961939
  'khmer': 1844759
  'huns': 1835466
  'ethiopians': 1809660
  'lithuanians': 1808825
  ... (46 more)

Total faction vocab size: 56 distinct values
NOTE: aoe2companion faction vocabulary is full civilization names (Franks, Mongols, etc.).
      Consumers MUST NOT treat faction as cross-dataset categorical without game-conditional encoding.


## Cell 14 -- Temporal sanity (I3)

Report min/max started_at, null count, distinct count.
TIMESTAMP pass-through from matches_raw.started (already TIMESTAMP).
No TRY_CAST -- no cast failures expected. Chronologically faithful ordering.

In [13]:
TEMPORAL_SANITY_SQL = """\
SELECT
    MIN(started_at)            AS min_started_at,
    MAX(started_at)            AS max_started_at,
    COUNT(*) FILTER (WHERE started_at IS NULL) AS null_started_at,
    COUNT(DISTINCT started_at) AS distinct_started_at
FROM matches_history_minimal\
"""

ts_row = con.execute(TEMPORAL_SANITY_SQL).fetchone()
min_started_at, max_started_at, null_started_at_ts, distinct_started_at = ts_row
print(f"min_started_at:      {min_started_at}")
print(f"max_started_at:      {max_started_at}")
print(f"null_started_at:     {null_started_at_ts}  (pass-through; expected 0)")
print(f"distinct_started_at: {distinct_started_at}")

min_started_at:      2020-07-31 23:30:34
max_started_at:      2026-04-04 23:58:58
null_started_at:     0  (pass-through; expected 0)
distinct_started_at: 26623674


## Cell 15 -- matchId range sanity (aoec-specific)

Report min/max matchId value and max decimal digit count.
aoec matchId is INTEGER (variable decimal width; 1-10 digits).
Exploratory only -- no gate. Confirms I7 numeric provenance.

In [14]:
MATCHID_RANGE_SQL = """\
SELECT
    MIN(CAST(split_part(match_id, '::', 2) AS BIGINT))              AS min_match_id_val,
    MAX(CAST(split_part(match_id, '::', 2) AS BIGINT))              AS max_match_id_val,
    MAX(length(split_part(match_id, '::', 2)))                      AS max_decimal_digits
FROM matches_history_minimal\
"""

mid_row = con.execute(MATCHID_RANGE_SQL).fetchone()
min_match_id_val, max_match_id_val, max_decimal_digits = mid_row
print(f"min_match_id_val:    {min_match_id_val}")
print(f"max_match_id_val:    {max_match_id_val}")
print(f"max_decimal_digits:  {max_decimal_digits}")
print("NOTE: Variable decimal width confirmed -- no fixed-length gate (aoec-specific).")

min_match_id_val:    32255750
max_match_id_val:    468020658
max_decimal_digits:  9
NOTE: Variable decimal width confirmed -- no fixed-length gate (aoec-specific).


## Cell 16 -- Build validation JSON + assert all_assertions_pass

Captures: step metadata, row_counts, assertion_results, sql_queries (verbatim I6),
describe_table_rows (DESCRIBE output for nullable-flag reproducibility).
Note: DuckDB may return DuckDBPyType for column_type; stringify with str(x).

In [15]:
reports_dir = get_reports_dir("aoe2", "aoe2companion")
artifact_dir = reports_dir / "artifacts" / "01_exploration" / "04_cleaning"
artifact_dir.mkdir(parents=True, exist_ok=True)

json_path = artifact_dir / "01_04_03_minimal_history_view.json"

# Capture DESCRIBE output as JSON-serializable list
describe_rows_raw = con.execute("DESCRIBE matches_history_minimal").fetchall()
describe_table_rows = [
    {
        "column_name": row[0],
        "column_type": str(row[1]),
        "null": row[2],
        "key": row[3],
        "default": row[4],
        "extra": row[5],
    }
    for row in describe_rows_raw
]

assertion_results = {
    "src_col_count_48": len(describe_src) == 48,
    "required_src_cols_present": all(c in src_col_names for c in required_cols),
    "col_count_8": len(view_col_names) == 8,
    "col_names_match": view_col_names == expected_col_names,
    "col_dtypes_match": view_col_types == expected_dtypes,
    "total_rows_61062392": total_rows == 61_062_392,
    "distinct_match_ids_30531196": distinct_match_ids == 30_531_196,
    "src_rows_61062392": src_rows == 61_062_392,
    "matches_with_not_2_rows_0": matches_with_not_2_rows == 0,
    "symmetry_violations_0": symmetry_violations == 0,
    "null_match_id_0": null_match_id == 0,
    "null_player_id_0": null_player_id == 0,
    "null_opponent_id_0": null_opponent_id == 0,
    "null_won_0": null_won == 0,
    "null_dataset_tag_0": null_dataset_tag == 0,
    "null_faction_0": null_faction == 0,
    "null_opponent_faction_0": null_opponent_faction == 0,
    "prefix_violations_0": prefix_violations == 0,
    "n_distinct_tags_1": n_distinct_tags == 1,
    "dataset_tag_aoe2companion": the_tag == "aoe2companion",
}

all_assertions_pass = all(assertion_results.values())

validation = {
    "step": "01_04_03",
    "dataset": "aoe2companion",
    "game": "aoe2",
    "generated_date": str(date.today()),
    "object": "matches_history_minimal",
    "object_type": "table",
    "implementation_note": (
        "DuckDB 1.5.1 bug: (1) self-join on matches_1v1_clean VIEW (row_number+ANY) "
        "causes InternalException; (2) QUALIFY+CTE self-join in a single CREATE TABLE "
        "gives wrong row counts (42,866 instead of 61,062,392). Workaround: three-step "
        "DDL -- (1) CREATE TABLE _good_match_ids from matches_raw QUALIFY; "
        "(2) CREATE TABLE _mhm_base = matches_raw INNER JOIN _good_match_ids QUALIFY; "
        "(3) CREATE TABLE matches_history_minimal = self-join _mhm_base; "
        "(4) DROP staging tables. Semantics identical to planned CREATE OR REPLACE VIEW. "
        "Same 8-column schema contract maintained."
    ),
    "row_counts": {
        "total_rows": total_rows,
        "distinct_match_ids": distinct_match_ids,
        "src_rows": src_rows,
        "matches_with_2_rows": matches_with_2_rows,
        "matches_with_not_2_rows": matches_with_not_2_rows,
    },
    "null_counts": {
        "null_match_id": null_match_id,
        "null_started_at": null_started_at,
        "null_player_id": null_player_id,
        "null_opponent_id": null_opponent_id,
        "null_won": null_won,
        "null_dataset_tag": null_dataset_tag,
        "null_faction": null_faction,
        "null_opponent_faction": null_opponent_faction,
    },
    "symmetry_violations": symmetry_violations,
    "prefix_violations": prefix_violations,
    "dataset_tag_distinct": n_distinct_tags,
    "dataset_tag_value": the_tag,
    "temporal_sanity": {
        "min_started_at": str(min_started_at),
        "max_started_at": str(max_started_at),
        "null_started_at": null_started_at_ts,
        "distinct_started_at": distinct_started_at,
    },
    "matchid_range": {
        "min_match_id_val": min_match_id_val,
        "max_match_id_val": max_match_id_val,
        "max_decimal_digits": max_decimal_digits,
    },
    "faction_vocab": faction_vocab,
    "schema_shape": {
        "col_names": view_col_names,
        "col_types": view_col_types,
    },
    "assertion_results": assertion_results,
    "all_assertions_pass": all_assertions_pass,
    "describe_table_rows": describe_table_rows,
    "sql_queries": {
        "CREATE_GOOD_MATCH_IDS_SQL": CREATE_GOOD_MATCH_IDS_SQL,
        "CREATE_STAGING_TABLE_SQL": CREATE_STAGING_TABLE_SQL,
        "CREATE_MATCHES_HISTORY_MINIMAL_SQL": CREATE_MATCHES_HISTORY_MINIMAL_SQL,
        "DROP_STAGING_TABLES_SQL": DROP_STAGING_TABLES_SQL,
        "ROW_COUNT_CHECK_SQL": ROW_COUNT_CHECK_SQL,
        "ROW_COUNT_SRC_SQL": ROW_COUNT_SRC_SQL,
        "ROW_COUNT_NOT_2_SQL": ROW_COUNT_NOT_2_SQL,
        "SYMMETRY_I5_ANALOG_SQL": SYMMETRY_I5_ANALOG_SQL,
        "ZERO_NULL_SQL": ZERO_NULL_SQL,
        "PREFIX_CHECK_SQL": PREFIX_CHECK_SQL,
        "DATASET_TAG_CHECK_SQL": DATASET_TAG_CHECK_SQL,
        "FACTION_VOCAB_SQL": FACTION_VOCAB_SQL,
        "TEMPORAL_SANITY_SQL": TEMPORAL_SANITY_SQL,
        "MATCHID_RANGE_SQL": MATCHID_RANGE_SQL,
    },
    "spec_schema": {
        "expected_col_names": expected_col_names,
        "expected_dtypes": expected_dtypes,
    },
}

with open(json_path, "w") as f:
    json.dump(validation, f, indent=2, default=str)

print(f"Validation JSON written: {json_path}")
print(f"All assertions pass: {all_assertions_pass}")

assert all_assertions_pass, (
    f"One or more assertions FAILED:\n"
    + "\n".join(f"  {k}: {v}" for k, v in assertion_results.items() if not v)
)

Validation JSON written: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2companion/reports/artifacts/01_exploration/04_cleaning/01_04_03_minimal_history_view.json
All assertions pass: True


## Cell 17 -- Build markdown report

In [16]:
md_path = artifact_dir / "01_04_03_minimal_history_view.md"

faction_table_rows = "\n".join(
    f"| `{faction}` | {n} |" for faction, n in faction_rows[:20]
)

md_content = f"""# Step 01_04_03 -- Minimal Cross-Dataset History View (aoe2companion)

**Generated:** {date.today()}
**Dataset:** aoe2companion
**Game:** AoE2
**Step:** 01_04_03
**Predecessor:** 01_04_02 (Data Cleaning Execution)

## Summary

Created `matches_history_minimal` TABLE -- 8-column player-row-grain projection of
`matches_raw` (filtered to matches_1v1_clean scope; 2 rows per 1v1 match). Self-join
pattern (sc2egset PR #152 precedent). Canonical TIMESTAMP temporal dtype (pass-through;
matches_raw.started is already TIMESTAMP). Per-dataset-polymorphic faction
vocabulary (full AoE2 civ names). Cross-dataset-harmonized substrate for Phase 02+
rating-system backtesting. Pure non-destructive projection (I9).

**Implementation note (DuckDB 1.5.1 workaround):**
The plan specified CREATE OR REPLACE VIEW. DuckDB 1.5.1 has a confirmed bug where
a self-join on a VIEW that uses row_number() + ANY() window functions (as in
matches_1v1_clean) causes InternalException or wrong row counts. Workaround: two-step
DDL -- (1) CREATE TABLE _mhm_base from matches_raw with same filter logic as
matches_1v1_clean; (2) CREATE TABLE matches_history_minimal via self-join of _mhm_base;
(3) DROP TABLE _mhm_base. Same 8-column contract, same row counts, same semantics.

aoec-specific notes:
- No UNION ALL pivot needed (natively 2-row per match).
- No slot-bias gate (no slot column; natively player-row).
- Zero-NULL gate for faction + opponent_faction (civ zero-NULL upstream per 01_04_02).
- matchId INTEGER -> variable decimal width; numeric-tail regex + round-trip cast (I7).

## Schema (8 columns)

| column | dtype | semantics |
|---|---|---|
| `match_id` | VARCHAR | `'aoe2companion::'` + decimal matchId (variable width) |
| `started_at` | TIMESTAMP | Pass-through from matches_raw.started (already TIMESTAMP) |
| `player_id` | VARCHAR | CAST(profileId AS VARCHAR) |
| `opponent_id` | VARCHAR | Opposing profileId (from self-join) |
| `faction` | VARCHAR | Full civ name (e.g., Franks, Mongols). PER-DATASET POLYMORPHIC |
| `opponent_faction` | VARCHAR | Opposing civ (same vocabulary as faction) |
| `won` | BOOLEAN | Focal player's outcome (complementary between the 2 rows) |
| `dataset_tag` | VARCHAR | Constant `'aoe2companion'` |

## Row-count flow

| metric | value |
|---|---|
| Source matches_1v1_clean rows | {src_rows} |
| matches_history_minimal total rows | {total_rows} |
| distinct match_ids | {distinct_match_ids} |
| matches with exactly 2 rows | {matches_with_2_rows} |
| matches with NOT 2 rows | {matches_with_not_2_rows} |

## matchId range (aoec-specific, exploratory)

| metric | value |
|---|---|
| min_match_id_val | {min_match_id_val} |
| max_match_id_val | {max_match_id_val} |
| max_decimal_digits | {max_decimal_digits} |

## Faction vocabulary (per-dataset polymorphic, top 20)

| faction | count |
|---|---|
{faction_table_rows}

NOTE: aoe2companion faction vocabulary is full civilization names (e.g., Franks, Mongols).
Consumers MUST NOT treat faction as a single categorical feature across
datasets without game-conditional encoding.

## Temporal sanity (I3)

| metric | value |
|---|---|
| min_started_at | {min_started_at} |
| max_started_at | {max_started_at} |
| null_started_at (pass-through) | {null_started_at_ts} |
| distinct_started_at | {distinct_started_at} |

## NULL counts

| column | null count | gate |
|---|---|---|
| match_id | {null_match_id} | 0 (GATE) |
| started_at | {null_started_at} | 0 (GATE; pass-through TIMESTAMP) |
| player_id | {null_player_id} | 0 (GATE) |
| opponent_id | {null_opponent_id} | 0 (GATE) |
| won | {null_won} | 0 (GATE) |
| dataset_tag | {null_dataset_tag} | 0 (GATE) |
| faction | {null_faction} | 0 (GATE; civ zero-NULL upstream) |
| opponent_faction | {null_opponent_faction} | 0 (GATE; civ zero-NULL upstream) |

## Gate verdict (12 gates; no slot-bias gate -- aoec natively player-row)

| check | result |
|---|---|
| Row count 61,062,392 = 2 x 30,531,196 | {'PASS' if total_rows == 61_062_392 and distinct_match_ids == 30_531_196 else 'FAIL'} |
| Column count 8 | {'PASS' if len(view_col_names) == 8 else 'FAIL'} |
| started_at dtype TIMESTAMP | {'PASS' if 'TIMESTAMP' in view_col_types[1] else 'FAIL'} |
| I5-analog NULL-safe symmetry violations (IS DISTINCT FROM) = 0 | {'PASS' if symmetry_violations == 0 else 'FAIL'} |
| Zero NULLs: match_id / player_id / opponent_id / won / dataset_tag | {'PASS' if null_match_id == null_player_id == null_opponent_id == null_won == null_dataset_tag == 0 else 'FAIL'} |
| Zero NULLs: faction / opponent_faction (civ zero-NULL upstream) | {'PASS' if null_faction == null_opponent_faction == 0 else 'FAIL'} |
| Prefix violations = 0 (numeric-tail regex + round-trip cast) | {'PASS' if prefix_violations == 0 else 'FAIL'} |
| dataset_tag distinct count = 1, value 'aoe2companion' | {'PASS' if n_distinct_tags == 1 and the_tag == 'aoe2companion' else 'FAIL'} |
| matches_with_not_2_rows = 0 | {'PASS' if matches_with_not_2_rows == 0 else 'FAIL'} |
| All assertions pass | {'PASS' if all_assertions_pass else 'FAIL'} |

## Artifact

Validation JSON: `{json_path.relative_to(json_path.parents[8])}`
"""

with open(md_path, "w") as f:
    f.write(md_content)

print(f"Markdown report written: {md_path}")

Markdown report written: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2companion/reports/artifacts/01_exploration/04_cleaning/01_04_03_minimal_history_view.md


## Cell 18 -- Write schema YAML for matches_history_minimal

Nullable flags sourced from DESCRIBE result (DuckDB DESCRIBE 6-tuple contract:
index 2 is the null flag: 'YES' -> nullable=True, 'NO' -> nullable=False).
Concrete Python booleans written -- no '<from DESCRIBE>' string literals.
Faction description includes the "per-dataset polymorphic vocabulary" warning (I8).
match_id description cites matchId INTEGER provenance (I7).
object_type: table (DuckDB 1.5.1 workaround; semantics identical to VIEW).

In [17]:
schema_dir = reports_dir.parent / "data" / "db" / "schemas" / "views"
schema_dir.mkdir(parents=True, exist_ok=True)

yaml_path = schema_dir / "matches_history_minimal.yaml"

# Translate DESCRIBE nullable flags to concrete Python booleans
describe_rows_for_yaml = con.execute("DESCRIBE matches_history_minimal").fetchall()
nullable_map = {row[0]: (row[2] == "YES") for row in describe_rows_for_yaml}

schema = {
    "table": "matches_history_minimal",
    "dataset": "aoe2companion",
    "game": "aoe2",
    "object_type": "table",
    "object_type_note": (
        "DuckDB 1.5.1 bug: self-join on matches_1v1_clean VIEW (row_number + ANY) "
        "causes InternalException. Two-step DDL materialization used instead of "
        "CREATE OR REPLACE VIEW. Semantics are identical to the planned VIEW -- same "
        "8-column contract, same filtering, same row count. Consumer code works "
        "identically for both TABLE and VIEW in DuckDB SQL."
    ),
    "step": "01_04_03",
    "row_count": total_rows,
    "describe_artifact": (
        "src/rts_predict/games/aoe2/datasets/aoe2companion/reports/artifacts"
        "/01_exploration/04_cleaning/01_04_03_minimal_history_view.json"
    ),
    "generated_date": str(date.today()),
    "columns": [
        {
            "name": "match_id",
            "type": "VARCHAR",
            "nullable": nullable_map["match_id"],
            "description": (
                "Cross-dataset unique match identifier. Format: '<dataset_tag>::<native_id>'. "
                "For aoe2companion: 'aoe2companion::<decimal-matchId>'. Prefix guarantees UNION ALL "
                "uniqueness across sibling datasets. Length is VARIABLE (aoec matchId is INTEGER, "
                "variable decimal width; no fixed-length gate like sc2egset's 42-char hex). "
                "Provenance: numeric-tail regex [0-9]+ cites "
                "src/rts_predict/games/aoe2/datasets/aoe2companion/data/db/schemas/raw/matches_raw.yaml "
                "line `matchId: INTEGER` (I7)."
            ),
            "notes": "IDENTITY. Prefix applied in DDL only; upstream matchId unchanged (I9).",
        },
        {
            "name": "started_at",
            "type": "TIMESTAMP",
            "nullable": nullable_map["started_at"],
            "description": (
                "Match start time. TIMESTAMP (no TZ) -- pass-through from "
                "matches_raw.started (already TIMESTAMP per "
                "data/db/schemas/raw/matches_raw.yaml). No cast. Canonical cross-dataset "
                "dtype: sibling objects (sc2egset TRY_CAST from VARCHAR; aoestats CAST from "
                "TIMESTAMPTZ AT TIME ZONE 'UTC') all emit TIMESTAMP."
            ),
            "notes": (
                "CONTEXT. Temporal anchor for Phase 02 rating-update loops. "
                "Chronologically faithful (TIMESTAMP ordering, not lex)."
            ),
        },
        {
            "name": "player_id",
            "type": "VARCHAR",
            "nullable": nullable_map["player_id"],
            "description": (
                "Focal player identifier. aoe2companion: CAST(profileId AS VARCHAR) "
                "(INTEGER -> VARCHAR)."
            ),
            "notes": (
                "IDENTITY. Per-dataset identifier; cross-dataset identity resolution "
                "is a future step (Phase 01_05+)."
            ),
        },
        {
            "name": "opponent_id",
            "type": "VARCHAR",
            "nullable": nullable_map["opponent_id"],
            "description": "Opposing player identifier. Same grain and provenance as player_id.",
            "notes": "IDENTITY.",
        },
        {
            "name": "faction",
            "type": "VARCHAR",
            "nullable": nullable_map["faction"],
            "description": (
                "Focal player's faction. Per-dataset polymorphic vocabulary (cross-dataset "
                "column name + dtype only -- values differ in ontology). sc2egset: 4-char race "
                "stems Prot/Terr/Zerg (empirically verified; NOT full 'Protoss'/'Terran'/'Zerg'). "
                "aoestats: full civilization names (Mongols, Franks, etc.). "
                "aoe2companion: full civilization names (from matches_raw.civ -- zero NULLs upstream). "
                "CONSUMERS MUST NOT treat faction as a single categorical feature across "
                "datasets without game-conditional encoding (e.g., WHERE dataset_tag = 'aoe2companion' "
                "before GROUP BY faction)."
            ),
            "notes": (
                "PRE_GAME. Raw vocabulary (civilization actually played). "
                "Polymorphic I8 contract -- see description. Zero NULLs (civ zero-NULL upstream)."
            ),
        },
        {
            "name": "opponent_faction",
            "type": "VARCHAR",
            "nullable": nullable_map["opponent_faction"],
            "description": "Opposing player's faction (same per-dataset vocabulary as `faction`).",
            "notes": "PRE_GAME. Mirror of faction from the opponent row. Zero NULLs (civ zero-NULL upstream).",
        },
        {
            "name": "won",
            "type": "BOOLEAN",
            "nullable": nullable_map["won"],
            "description": (
                "TRUE if the focal player won, FALSE otherwise. The two rows of a match have "
                "complementary `won` values (exactly one TRUE, one FALSE)."
            ),
            "notes": (
                "TARGET. Pass-through from matches_raw.won; prediction label "
                "for downstream experiments. Zero NULLs (R03 complementarity filter upstream)."
            ),
        },
        {
            "name": "dataset_tag",
            "type": "VARCHAR",
            "nullable": False,
            "description": (
                "Dataset discriminator for UNION ALL across sibling datasets. "
                "Constant 'aoe2companion' in this TABLE."
            ),
            "notes": "IDENTITY. Matches the prefix before '::' in match_id.",
        },
    ],
    "provenance": {
        "source_tables": ["matches_raw"],
        "source_note": (
            "Reads from matches_raw directly (not matches_1v1_clean VIEW) due to DuckDB 1.5.1 "
            "bug with self-join on VIEWs containing window functions. Applies identical filter "
            "logic: internalLeaderboardId IN (6, 18); profileId != -1; deduplicated by "
            "(matchId, profileId) ORDER BY started; R03 complementarity (HAVING COUNT(*)=2 "
            "AND n_true=1 AND n_false=1)."
        ),
        "join_key": (
            "self-join on 'aoe2companion::' || CAST(matchId AS VARCHAR); "
            "player_id = CAST(profileId AS VARCHAR); "
            "opponent_id from sibling row where profileId <> opponent profileId"
        ),
        "filter": (
            "internalLeaderboardId IN (6, 18); profileId != -1; "
            "deduplicated; R03 complementarity (HAVING COUNT(*)=2 AND n_true=1 AND n_false=1)."
        ),
        "scope": (
            "30,531,196 true 1v1 decisive matches, 61,062,392 player-rows. "
            "Cross-dataset harmonization substrate for Phase 02+ rating backtesting."
        ),
        "created_by": (
            "sandbox/aoe2/aoe2companion/01_exploration/04_cleaning/"
            "01_04_03_minimal_history_view.py"
        ),
    },
    "invariants": [
        {
            "id": "I3",
            "description": (
                "TIMESTAMP-typed temporal anchor. started_at is a pass-through from "
                "matches_raw.started (already TIMESTAMP per matches_raw.yaml). "
                "No TRY_CAST needed. Chronologically faithful ordering. No windowed "
                "aggregations, no shift(), no future joins. Phase 02 consumers use "
                "started_at as the strict-less-than anchor for match_time < T feature computation."
            ),
        },
        {
            "id": "I5",
            "description": (
                "Player-row symmetry (I5-analog). Every match_id has exactly 2 rows. "
                "(player_id, opponent_id) appears once in each direction. The two `won` "
                "values are complementary. faction and opponent_faction are mirror "
                "images. Assertion SQL uses IS DISTINCT FROM for NULL-safe comparison. "
                "Slot-bias gate is N/A for aoec (natively player-row; no slot column)."
            ),
        },
        {
            "id": "I6",
            "description": (
                "CREATE TABLE DDL + every assertion SQL stored verbatim in "
                "01_04_03_minimal_history_view.json sql_queries block. DESCRIBE result "
                "captured in validation JSON describe_table_rows for reproducibility "
                "of the nullable flags written to this YAML."
            ),
        },
        {
            "id": "I7",
            "description": (
                "Magic literal `[0-9]+` numeric-tail regex in PREFIX_CHECK_SQL cites "
                "src/rts_predict/games/aoe2/datasets/aoe2companion/data/db/schemas/raw/"
                "matches_raw.yaml line `matchId: INTEGER`. Round-trip BIGINT cast "
                "verifies decimal integrity. Variable decimal width -- no fixed-length gate."
            ),
        },
        {
            "id": "I8",
            "description": (
                "Cross-dataset comparability: 8-column names + dtypes are the cross-"
                "dataset contract. Canonical temporal dtype = TIMESTAMP (no TZ). Faction "
                "vocabulary is per-dataset-polymorphic -- column name and dtype cross-"
                "dataset, values per-dataset ontology. aoe2companion uses full civ names "
                "(vs sc2egset 4-char stems). match_id prefixed 'aoe2companion::'."
            ),
        },
        {
            "id": "I9",
            "description": (
                "Pure non-destructive projection. No raw table modified. matches_raw "
                "is read-only (SELECT only). matches_1v1_clean VIEW unchanged (not used "
                "in DDL due to DuckDB 1.5.1 bug, but DESCRIBE still confirms its schema). "
                "Staging TABLE _mhm_base created and immediately dropped. Only "
                "matches_history_minimal TABLE is the persistent artifact."
            ),
        },
    ],
    "provenance_categories_note": (
        "This table inherits provenance categories from matches_raw (applying "
        "matches_1v1_clean filter logic). Per-column 'notes' field uses the "
        "single-token vocabulary (IDENTITY, CONTEXT, PRE_GAME, TARGET) established "
        "in 01_04_02."
    ),
}

with open(yaml_path, "w") as f:
    yaml.safe_dump(schema, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print(f"Schema YAML written: {yaml_path}")
print(f"Nullable values from DESCRIBE:")
for col_name, nullable_val in nullable_map.items():
    print(f"  {col_name}: nullable={nullable_val}")

Schema YAML written: /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2companion/data/db/schemas/views/matches_history_minimal.yaml
Nullable values from DESCRIBE:
  match_id: nullable=True
  started_at: nullable=True
  player_id: nullable=True
  opponent_id: nullable=True
  faction: nullable=True
  opponent_faction: nullable=True
  won: nullable=True
  dataset_tag: nullable=True


## Cell 19 -- Close connection + final summary

In [18]:
db.close()
print("DuckDB connection closed.")

print("\n=== FINAL SUMMARY: Step 01_04_03 (aoe2companion) ===")
print(f"TABLE created:         matches_history_minimal")
print(f"Rows:                  {total_rows} ({distinct_match_ids} matches x 2)")
print(f"Schema:                {len(view_col_names)} cols, started_at dtype: {view_col_types[1]}")
print(f"Symmetry violations:   {symmetry_violations}")
print(f"Prefix violations:     {prefix_violations}")
print(f"dataset_tag distinct:  {n_distinct_tags}")
print(f"Faction vocab size:    {len(faction_vocab)} distinct civs")
print(f"Faction vocab (top 5): {list(faction_vocab.keys())[:5]}")
print(f"matchId range:         {min_match_id_val} to {max_match_id_val} (max {max_decimal_digits} digits)")
print(f"All assertions pass:   {all_assertions_pass}")
print(f"\nArtifacts:")
print(f"  {json_path}")
print(f"  {md_path}")
print(f"  {yaml_path}")

DuckDB connection closed.

=== FINAL SUMMARY: Step 01_04_03 (aoe2companion) ===
TABLE created:         matches_history_minimal
Rows:                  61062392 (30531196 matches x 2)
Schema:                8 cols, started_at dtype: TIMESTAMP
Symmetry violations:   0
Prefix violations:     0
dataset_tag distinct:  1
Faction vocab size:    56 distinct civs
Faction vocab (top 5): ['franks', 'mongols', 'britons', 'magyars', 'spanish']
matchId range:         32255750 to 468020658 (max 9 digits)
All assertions pass:   True

Artifacts:
  /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2companion/reports/artifacts/01_exploration/04_cleaning/01_04_03_minimal_history_view.json
  /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2companion/reports/artifacts/01_exploration/04_cleaning/01_04_03_minimal_history_view.md
  /Users/tomaszpionka/Projects/rts-outcome-prediction/src/rts_predict/games/aoe2/datasets/aoe2compan